# Conditional Workflow: Quadratic Equation Solver

This notebook demonstrates **conditional branching (`add_conditional_edges`)** in LangGraph:

```text
START -> show_equation -> cal_discriminant
                               |
                 (check discriminant D = b^2 - 4ac)
                /              |              \
           (D > 0)          (D == 0)        (D < 0)
             v                 v              v
         real_roots     repeated_roots  no_real_roots
             \                 |              /
              ----->----->--- END <----<-----
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

In [ ]:
# 1. Define State
class QuadState(TypedDict):
    a: int
    b: int 
    c: int

    equation: str
    discriminant: float
    result: str

In [ ]:
# 2. Node Functions

def show_equation(state: QuadState):
    equation = f"{state['a']}x^2 + ({state['b']})x + ({state['c']}) = 0"
    return {'equation': equation}

def cal_discriminant(state: QuadState):
    discriminant = state['b']**2 - (4 * state['a'] * state['c'])
    return {'discriminant': discriminant}

# Note: parameter must be `state: QuadState` (not `QuadState`)
# Note: math is (2 * state['a']), no curly braces!
def real_roots(state: QuadState):
    d = state["discriminant"]
    root1 = (-state["b"] + d**0.5) / (2 * state["a"])
    root2 = (-state["b"] - d**0.5) / (2 * state["a"])
    result = f"Two distinct real roots: {root1:.2f} and {root2:.2f}"
    return {'result': result}

def repeated_roots(state: QuadState):
    root = (-state["b"]) / (2 * state["a"])
    result = f"One repeated real root: {root:.2f}"
    return {'result': result}

def no_real_roots(state: QuadState):
    d = state["discriminant"]
    real_part = -state["b"] / (2 * state["a"])
    imag_part = ((-d)**0.5) / (2 * state["a"])
    result = f"No real roots (Complex roots): {real_part:.2f} ± {imag_part:.2f}i"
    return {'result': result}

In [ ]:
# 3. Conditional Routing Function
def check_roots(state: QuadState) -> Literal["real_roots", "repeated_roots", "no_real_roots"]:
    d = state["discriminant"]
    if d > 0:
        return "real_roots"
    elif d == 0:
        return "repeated_roots"
    else:
        return "no_real_roots"

In [ ]:
# 4. Build Conditional Graph
graph = StateGraph(QuadState)

# Add nodes
graph.add_node('show_equation', show_equation)
graph.add_node('cal_discriminant', cal_discriminant)
graph.add_node('real_roots', real_roots)
graph.add_node('repeated_roots', repeated_roots)
graph.add_node('no_real_roots', no_real_roots)

# Regular edges
graph.add_edge(START, 'show_equation')
graph.add_edge('show_equation', 'cal_discriminant')

# Conditional routing edge from cal_discriminant
graph.add_conditional_edges(
    'cal_discriminant',
    check_roots,
    {
        'real_roots': 'real_roots',
        'repeated_roots': 'repeated_roots',
        'no_real_roots': 'no_real_roots'
    }
)

# Edges to END
graph.add_edge('real_roots', END)
graph.add_edge('repeated_roots', END)
graph.add_edge('no_real_roots', END)

# Compile
workflow = graph.compile()

In [ ]:
# Visualize workflow graph
from IPython.display import Image, display
try:
    display(Image(workflow.get_graph().draw_mermaid_png()))
except Exception:
    print(workflow.get_graph().draw_ascii())

In [ ]:
# Test Case 1: Complex Roots (D < 0): 4x^2 - 5x + 6 = 0
res1 = workflow.invoke({'a': 4, 'b': -5, 'c': 6})
print("Equation:", res1['equation'])
print("Discriminant:", res1['discriminant'])
print("Result:", res1['result'])
print("-" * 50)

# Test Case 2: Two Real Roots (D > 0): 1x^2 - 5x + 6 = 0
res2 = workflow.invoke({'a': 1, 'b': -5, 'c': 6})
print("Equation:", res2['equation'])
print("Discriminant:", res2['discriminant'])
print("Result:", res2['result'])
print("-" * 50)

# Test Case 3: One Repeated Root (D = 0): 1x^2 - 4x + 4 = 0
res3 = workflow.invoke({'a': 1, 'b': -4, 'c': 4})
print("Equation:", res3['equation'])
print("Discriminant:", res3['discriminant'])
print("Result:", res3['result'])